# Historian Events Loader

This notebook processes the historian and events data for a given alarm tag.

**Workflow:**
1. Load the alarm tag's related-tag information from the Excel workbook (`UC3_Alarm_Tags_Presentation_30MAY2026.xlsx`).
2. Extract DCS tag names and their associated asset IDs from the sheet.
3. Map those short asset IDs (e.g. `1E`, `1F`) to UUIDs using the `ADNOC-B.json` config file.
4. Discover which UUID folders exist under `Historian_Events/Historian/` and `Historian_Events/events/`.
5. Load the historian time-series parquet files for each related tag.
6. Load the event parquet files (_E, _EA, _EAL) for each asset.

**Target tag for this run:** `03TIC_1023` (sheet: `03TIC_1023 PVLO_PVHI`)

## 0. Configuration

In [1]:
# ── Paths ──────────────────────────────────────────────────────────────────────
import os

REPO_ROOT          = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '.'))
DATA_DIR           = os.path.join(REPO_ROOT, 'DATA')
HISTORIAN_ROOT     = os.path.join(DATA_DIR, 'Historian_Events')
HISTORIAN_DIR      = os.path.join(HISTORIAN_ROOT, 'Historian')   # tag time-series
EVENTS_DIR         = os.path.join(HISTORIAN_ROOT, 'events')       # alarm/event tables
EXCEL_FILE         = os.path.join(DATA_DIR, 'UC3_Alarm_Tags_Presentation_30MAY2026.xlsx')
ADNOC_JSON         = os.path.join(DATA_DIR, 'config', 'config', 'EMDB', 'ADNOC-B.json')

# ── Target alarm tag ──────────────────────────────────────────────────────────
TARGET_TAG         = '03PIC_1104'
EXCEL_HEADER_ROW   = 6      # 0-indexed row that contains column headers

# ── Per-tag asset ID overrides ─────────────────────────────────────────────────
# If a tag's assets are known and should not be auto-derived from the Excel sheet,
# list them explicitly here. Keys are TARGET_TAG values.
ASSET_ID_OVERRIDE = {
    '03LIC_1071': ['1E', '1F', '1I', '1K', '1L'],
}

# Auto-detect the sheet whose name contains TARGET_TAG
import openpyxl
_wb = openpyxl.load_workbook(EXCEL_FILE, read_only=True, data_only=True)
_matches = [s for s in _wb.sheetnames if TARGET_TAG in s]
_wb.close()
if not _matches:
    raise ValueError(f"No sheet found containing '{TARGET_TAG}' in {EXCEL_FILE}")
SHEET_NAME = _matches[0]


print(f"REPO_ROOT      : {REPO_ROOT}")
print(f"SHEET_NAME     : {SHEET_NAME}  (auto-detected)")

print(f"EXCEL_FILE     : {EXCEL_FILE}")
print(f"TARGET_TAG     : {TARGET_TAG}")

print(f"ADNOC_JSON     : {ADNOC_JSON}")
print(f"EVENTS_DIR     : {EVENTS_DIR}")
print(f"HISTORIAN_DIR  : {HISTORIAN_DIR}")

REPO_ROOT      : /home/h604827/ControlActions
SHEET_NAME     : 03PIC_1104 PVHI  (auto-detected)
EXCEL_FILE     : /home/h604827/ControlActions/DATA/UC3_Alarm_Tags_Presentation_30MAY2026.xlsx
TARGET_TAG     : 03PIC_1104
ADNOC_JSON     : /home/h604827/ControlActions/DATA/config/config/EMDB/ADNOC-B.json
EVENTS_DIR     : /home/h604827/ControlActions/DATA/Historian_Events/events
HISTORIAN_DIR  : /home/h604827/ControlActions/DATA/Historian_Events/Historian


## 1. Imports

In [2]:
import json
import pandas as pd

print(f"pandas  : {pd.__version__}")

pandas  : 2.3.3


## 2. Load Related-Tag Information from Excel

Each sheet in the Excel workbook describes the related tags for one alarm.  
We extract **DCS Tag Name**, **DESCRIPTION**, **PARAMETER**, **IS DATA AVAILABLE?** and **ASSET ID**.

In [3]:
print(f"Reading sheet '{SHEET_NAME}' from:\n  {EXCEL_FILE}\n")

df_sheet = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=EXCEL_HEADER_ROW)

print(f"Raw sheet shape  : {df_sheet.shape}")
print(f"Columns ({len(df_sheet.columns)}):")
for c in df_sheet.columns:
    print(f"  {c}")

Reading sheet '03PIC_1104 PVHI' from:
  /home/h604827/ControlActions/DATA/UC3_Alarm_Tags_Presentation_30MAY2026.xlsx

Raw sheet shape  : (55, 27)
Columns (27):
  Remarks
  Potential Input parameter
  Description
  Causes
  Plant
  P&ID
  SME Comments
  Unnamed: 7
  Unnamed: 8
  MH Comments
  DCS Tag Name
  DESCRIPTION
  PARAMETER
  IS DATA AVAILABLE?
  SP-HIGH LIMIT
  SP-LOW LIMIT
  EXTENDED PV HIGH LIMIT
  PV-HIGH LIMIT
  EXTENDED PV LOW LIMIT
  PV-LOW LIMIT
  OP-HIGH LIMIT
  OP-LOW LIMIT
  PHD_TAG AVILABLE
  FILE NAME
  ASSET ID
  AVAILABLE IN APC
  SAFETY MANAGER TAGS


In [4]:
# Keep only rows that have a DCS Tag Name
KEEP_COLS = ['Potential Input parameter', 'DCS Tag Name', 'DESCRIPTION', 'PARAMETER',
             'IS DATA AVAILABLE?', 'ASSET ID']

df_tags = (
    df_sheet[KEEP_COLS]
    .dropna(subset=['DCS Tag Name'])
    .copy()
)

# Normalise strings
df_tags['DCS Tag Name'] = df_tags['DCS Tag Name'].str.strip()
df_tags['ASSET ID']     = df_tags['ASSET ID'].astype(str).str.strip()

# Reset index for clean display
df_tags = df_tags.reset_index(drop=True)

print(f"Related tags found: {len(df_tags)}")
print()
display(df_tags)

Related tags found: 36



,Potential Input parameter,DCS Tag Name,DESCRIPTION,PARAMETER,IS DATA AVAILABLE?,ASSET ID
0,03TI_1408,03TI_1408,3E105 GAS TO 3E103 TEMP,PV,YES,1F
1,03SDV_3198,03SDV_3198,3C152 GAS BYPASS VALV,PV,YES,1K
2,03SDV_3197,03SDV_3197,3C152 GAS INLET SDV,PV,YES,1K
3,03PIC_1068,03PIC_1068,3C102 OVHD PRES,PV,YES,1F
4,03PIC_1104,03PIC_1104,3C105 OVHD GAS PRES,PV,YES,1F
5,03TI_1427,03TI_1427,3C105 OVHD GAS TEMP,PV,YES,1F
6,03TI_1099,03TI_1099,3C105 TRAY#11 TEMP,PV,YES,1F
7,03TI_1100,03TI_1100,3C105 TRAY# 7 TEMP,PV,YES,1F
8,03TI_1101,03TI_1101,3C105 TRAY# 3 TEMP,PV,YES,1F
9,03TI_1102,03TI_1102,3C105 TRAY#1 TEMP,PV,YES,1F


## 3. Load ADNOC-B.json and Build Asset-ID → UUID Mapping

The JSON config contains one entry per asset node in the ADNOC Buhasa plant hierarchy.  
Leaf nodes whose `Name` matches the short asset ID in the sheet provide the UUID we need.

In [5]:
print(f"Reading asset config:\n  {ADNOC_JSON}\n")

with open(ADNOC_JSON, 'r') as f:
    asset_nodes = json.load(f)

print(f"Total nodes in ADNOC-B.json : {len(asset_nodes)}")

# Build a flat map:  short-name (e.g. '1E') → UUID
# We only care about leaf nodes under TRAIN_1 (adjust if other areas are needed)
asset_id_to_uuid = {}
for node in asset_nodes:
    asset_id_to_uuid[node['Name']] = {
        'uuid'      : node['Id'],
        'hierarchy' : node['Hierarchy'],
        'leaf'      : node['LeafNode'],
    }

print(f"Unique asset names mapped    : {len(asset_id_to_uuid)}")
print()

# Show all leaf nodes for easy inspection
leaf_map = pd.DataFrame(
    [{'Asset Name': k, 'UUID': v['uuid'], 'Hierarchy': v['hierarchy'], 'IsLeaf': v['leaf']}
     for k, v in asset_id_to_uuid.items()]
).sort_values('Asset Name').reset_index(drop=True)

print("Full asset map (all nodes):")
display(leaf_map)

Reading asset config:
  /home/h604827/ControlActions/DATA/config/config/EMDB/ADNOC-B.json

Total nodes in ADNOC-B.json : 74
Unique asset names mapped    : 74

Full asset map (all nodes):


,Asset Name,UUID,Hierarchy,IsLeaf
0,1A,8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1A,True
1,1B,f0c70dd0-139e-4102-b75d-77bccee3ca83,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1B,True
2,1C,c71817e8-5df3-4ad6-89fb-0b71ec7d9f96,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1C,True
3,1D,9cac06f4-131b-4af0-aea0-17b223973d12,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1D,True
4,1E,072404dc-ae93-4239-9f6b-f4624391c391,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1E,True
...,...,...,...,...
69,FIRE & GAS,de29a368-6e72-48a4-bbd2-84c32d2ac510,ADNOC\BUHASA\ADNOC-B\FIRE & GAS,False
70,LEAN_GAS,a663c56b-aafd-4a6a-837f-c8677655e9ca,ADNOC\BUHASA\ADNOC-B\LEAN_GAS,False
71,TRAIN_1,cdfe1dc7-d3db-4efe-a9b6-c5dec2902ccb,ADNOC\BUHASA\ADNOC-B\TRAIN_1,False
72,TRAIN_2,cc6268d1-0fe2-4fc2-8d84-79f6080b172d,ADNOC\BUHASA\ADNOC-B\TRAIN_2,False


## 4. Map Sheet Asset IDs → UUIDs

For every unique asset ID referenced in the sheet, look up the corresponding UUID.

In [6]:
if TARGET_TAG in ASSET_ID_OVERRIDE:
    unique_assets = ASSET_ID_OVERRIDE[TARGET_TAG]
    print(f"NOTE: Using hardcoded asset override for {TARGET_TAG}: {unique_assets}")
else:
    unique_assets = sorted(df_tags['ASSET ID'].dropna().unique())
print(f"Unique ASSET IDs in sheet : {unique_assets}")
print()

mapping_rows = []
for asset_name in unique_assets:
    if asset_name in asset_id_to_uuid:
        info = asset_id_to_uuid[asset_name]
        mapping_rows.append({
            'Asset Name' : asset_name,
            'UUID'       : info['uuid'],
            'Hierarchy'  : info['hierarchy'],
        })
        print(f"  {asset_name:6s}  →  {info['uuid']}  ({info['hierarchy']})")
    else:
        mapping_rows.append({'Asset Name': asset_name, 'UUID': None, 'Hierarchy': None})
        print(f"  {asset_name:6s}  →  [NOT FOUND in ADNOC-B.json]")

df_asset_map = pd.DataFrame(mapping_rows)
print(f"\nResolved: {df_asset_map['UUID'].notna().sum()} / {len(df_asset_map)} assets")


Unique ASSET IDs in sheet : ['1A', '1F', '1H', '1I', '1J', '1K', '1L', '3I']

  1A      →  8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1A)
  1F      →  ccfff937-ff8c-4f84-a31c-a41f00fa0ff8  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1F)
  1H      →  f84c869c-4c09-4485-9cdd-35a994fc2350  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1H)
  1I      →  0b4d6b2d-7ed9-42ad-8ed6-6ee2bd33172c  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1I)
  1J      →  49fe1f01-8f3d-4383-9a36-0d77de120771  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1J)
  1K      →  6024200f-19c3-422a-a0e3-695a29031803  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1K)
  1L      →  65f25414-8750-4713-b7a5-d073ee37c3d5  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1L)
  3I      →  e0f2c384-4118-4662-99a4-4957e28ae60e  (ADNOC\BUHASA\ADNOC-B\TRAIN_2\3I)

Resolved: 8 / 8 assets


In [7]:
# Attach UUID back to each row in df_tags for convenience
df_tags = df_tags.merge(
    df_asset_map.rename(columns={'Asset Name': 'ASSET ID'})[['ASSET ID', 'UUID']],
    on='ASSET ID', how='left'
)

print("df_tags enriched with UUID column:")
display(df_tags[['DCS Tag Name', 'DESCRIPTION', 'ASSET ID', 'UUID']])

df_tags enriched with UUID column:


,DCS Tag Name,DESCRIPTION,ASSET ID,UUID
0,03TI_1408,3E105 GAS TO 3E103 TEMP,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
1,03SDV_3198,3C152 GAS BYPASS VALV,1K,6024200f-19c3-422a-a0e3-695a29031803
2,03SDV_3197,3C152 GAS INLET SDV,1K,6024200f-19c3-422a-a0e3-695a29031803
3,03PIC_1068,3C102 OVHD PRES,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
4,03PIC_1104,3C105 OVHD GAS PRES,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
5,03TI_1427,3C105 OVHD GAS TEMP,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
6,03TI_1099,3C105 TRAY#11 TEMP,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
7,03TI_1100,3C105 TRAY# 7 TEMP,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
8,03TI_1101,3C105 TRAY# 3 TEMP,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
9,03TI_1102,3C105 TRAY#1 TEMP,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8


## 5. Discover Available Folders in Historian_Events

The `Historian/` sub-folder holds **tag-level time-series** parquet files (one file per tag per asset).  
The `events/` sub-folder holds **alarm & event** parquet files for each asset.

We check which asset UUID folders actually exist on disk.

In [8]:
historian_folders = set(os.listdir(HISTORIAN_DIR))
events_folders    = set(os.listdir(EVENTS_DIR))

print(f"Folders found in Historian/ : {len(historian_folders)}")
print(f"Folders found in events/    : {len(events_folders)}")
print()

availability = []
for _, row in df_asset_map.iterrows():
    uuid = row['UUID']
    in_hist   = uuid in historian_folders if uuid else False
    in_events = uuid in events_folders    if uuid else False
    availability.append({
        'Asset Name'      : row['Asset Name'],
        'UUID'            : uuid,
        'In Historian/'   : in_hist,
        'In events/'      : in_events,
    })
    status = []
    if in_hist:   status.append('Historian/')
    if in_events: status.append('events/')
    status_str = ', '.join(status) if status else 'MISSING in both'
    print(f"  {row['Asset Name']:6s}  {uuid}  →  {status_str}")

df_availability = pd.DataFrame(availability)
print()
display(df_availability)

Folders found in Historian/ : 13
Folders found in events/    : 70

  1A      8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3  →  Historian/, events/
  1F      ccfff937-ff8c-4f84-a31c-a41f00fa0ff8  →  Historian/, events/
  1H      f84c869c-4c09-4485-9cdd-35a994fc2350  →  Historian/, events/
  1I      0b4d6b2d-7ed9-42ad-8ed6-6ee2bd33172c  →  Historian/, events/
  1J      49fe1f01-8f3d-4383-9a36-0d77de120771  →  Historian/, events/
  1K      6024200f-19c3-422a-a0e3-695a29031803  →  Historian/, events/
  1L      65f25414-8750-4713-b7a5-d073ee37c3d5  →  Historian/, events/
  3I      e0f2c384-4118-4662-99a4-4957e28ae60e  →  Historian/, events/



,Asset Name,UUID,In Historian/,In events/
0,1A,8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3,True,True
1,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8,True,True
2,1H,f84c869c-4c09-4485-9cdd-35a994fc2350,True,True
3,1I,0b4d6b2d-7ed9-42ad-8ed6-6ee2bd33172c,True,True
4,1J,49fe1f01-8f3d-4383-9a36-0d77de120771,True,True
5,1K,6024200f-19c3-422a-a0e3-695a29031803,True,True
6,1L,65f25414-8750-4713-b7a5-d073ee37c3d5,True,True
7,3I,e0f2c384-4118-4662-99a4-4957e28ae60e,True,True


## 6. Load Historian Time-Series Data

For each asset that has a folder in `Historian/`, list the available tag files and load  
the ones corresponding to the tags referenced in the sheet.

Each parquet file in `Historian/<UUID>/` is named `<TAG>.<PARAMETER>` (e.g. `03TIC_1023.PV`).  
Columns: `startdatetime`, `enddatetime`, `timestamp`, `value`, `S`.

In [9]:
historian_data = {}   # key: 'TAG.PARAM'  →  value: DataFrame

for _, asset_row in df_asset_map.iterrows():
    uuid       = asset_row['UUID']
    asset_name = asset_row['Asset Name']

    if not uuid or uuid not in historian_folders:
        print(f"[SKIP] {asset_name} — no Historian folder")
        continue

    asset_hist_dir = os.path.join(HISTORIAN_DIR, uuid)
    available_files = os.listdir(asset_hist_dir)

    # Tags in the sheet that belong to this asset
    asset_tags = df_tags.loc[df_tags['UUID'] == uuid, 'DCS Tag Name'].tolist()

    print(f"\n── Asset {asset_name} ({uuid}) ──")
    print(f"   Files in Historian folder : {len(available_files)}")
    print(f"   Tags in sheet for asset   : {asset_tags}")

    for tag in asset_tags:
        param = df_tags.loc[df_tags['DCS Tag Name'] == tag, 'PARAMETER'].values
        param = param[0] if len(param) > 0 else 'PV'
        filename = f"{tag}.{param}"

        filepath = os.path.join(asset_hist_dir, filename)
        if os.path.exists(filepath):
            df_hist = pd.read_parquet(filepath)
            df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
            historian_data[filename] = df_hist
            print(f"   ✓ Loaded  {filename:30s}  →  shape {df_hist.shape}  "
                  f"| range: {df_hist['timestamp'].min()} … {df_hist['timestamp'].max()}")
        else:
            # Try scanning available files for a match
            matches = [f for f in available_files if f.startswith(tag + '.')]
            if matches:
                for m in matches:
                    df_hist = pd.read_parquet(os.path.join(asset_hist_dir, m))
                    df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
                    historian_data[m] = df_hist
                    print(f"   ✓ Loaded  {m:30s}  →  shape {df_hist.shape}  "
                          f"| range: {df_hist['timestamp'].min()} … {df_hist['timestamp'].max()}")
            else:
                print(f"   ✗ Not found: {filename}")

print(f"\n{'='*60}")
print(f"Total historian series loaded: {len(historian_data)}")


── Asset 1A (8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3) ──
   Files in Historian folder : 11
   Tags in sheet for asset   : ['02FI_1000']
   ✓ Loaded  02FI_1000.PV                    →  shape (2130628, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00

── Asset 1F (ccfff937-ff8c-4f84-a31c-a41f00fa0ff8) ──
   Files in Historian folder : 50
   Tags in sheet for asset   : ['03TI_1408', '03PIC_1068', '03PIC_1104', '03TI_1427', '03TI_1099', '03TI_1100', '03TI_1101', '03TI_1102', '03LIC_1071', '03FIC_1085', '03LIC_1094', '03GM_0114_R', '03GM_0114A_R', '03LIC_1097', '03FI_1105']
   ✓ Loaded  03TI_1408.PV                    →  shape (2169302, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00
   ✓ Loaded  03PIC_1068.PV                   →  shape (2169275, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00
   ✓ Loaded  03PIC_1104.PV                   →  shape (2169275, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00
   ✓ Loaded  03TI_1427.PV                    →  shape (2169291,

### 6a. Historian Data — Preview

In [10]:
for key, df_h in historian_data.items():
    print(f"\n── {key} ──")
    print(f"   Shape   : {df_h.shape}")
    print(f"   Columns : {df_h.columns.tolist()}")
    print(f"   Null %  : {(df_h['value'].isna().mean()*100):.2f}%  missing values")
    display(df_h.head(3))


── 02FI_1000.PV ──
   Shape   : (2130628, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,6.623204,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,6.683493,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,6.603747,NaN



── 03TI_1408.PV ──
   Shape   : (2169302, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 96.77%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,15.125644,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,15.075137,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,15.022305,NaN



── 03PIC_1068.PV ──
   Shape   : (2169275, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 96.77%  missing values


...

,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,26.140095,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,26.329515,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,26.600484,NaN



── 03FI_1141A.PV ──
   Shape   : (2096863, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.35%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,64246.638467,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,64965.659267,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,64371.866200,NaN



── 03FIC_1474.PV ──
   Shape   : (2108675, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,232.780198,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,233.813448,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,230.480291,NaN


## 7. Load Events Data

For each asset UUID folder in `events/`, there are up to three types of parquet files:

| Suffix | Content |
|--------|---------|
| `_E`   | Process/operator change events |
| `_EA`  | Alarm events (no last/next linkage) |
| `_EAL` | Alarm events **with** linked previous/next alarm (includes duration) |

We load all three types for every relevant asset.

In [11]:
events_data = {}   # key: '<asset_name>_<type>'  →  value: DataFrame

for _, asset_row in df_asset_map.iterrows():
    uuid       = asset_row['UUID']
    asset_name = asset_row['Asset Name']

    if not uuid or uuid not in events_folders:
        print(f"[SKIP] {asset_name} — no events folder")
        continue

    asset_events_dir = os.path.join(EVENTS_DIR, uuid)
    available_files  = os.listdir(asset_events_dir)

    print(f"\n── Asset {asset_name} ({uuid}) ──")
    print(f"   Files in events folder : {available_files}")

    for suffix, label in [('_E.parquet', 'ChangeEvents'), ('_EA.parquet', 'AlarmEvents'), ('_EAL.parquet', 'AlarmEventsLinked')]:
        match = [f for f in available_files if f.endswith(suffix)]
        if match:
            fp = os.path.join(asset_events_dir, match[0])
            df_ev = pd.read_parquet(fp)
            # Parse timestamp column where available — use format='mixed' to handle
            # rows where milliseconds are present/absent in the same column
            for ts_col in ['VT_Start', 'Time', 'timestamp']:
                if ts_col in df_ev.columns:
                    df_ev[ts_col] = pd.to_datetime(df_ev[ts_col], format='mixed')
            key = f"{asset_name}_{label}"
            events_data[key] = df_ev
            print(f"   ✓ Loaded  {match[0]:50s}  →  shape {df_ev.shape}")
        else:
            print(f"   ✗ No {suffix} file found")

print(f"\n{'='*60}")
print(f"Total event tables loaded: {len(events_data)}")


── Asset 1A (8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3) ──
   Files in events folder : ['8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3.parquet', '2021011814_2026011001_EAE.parquet', '2021011814_2026011001_EA.parquet', '2021011814_2026011001_EALE.parquet', '2021011814_2026011001_EAL.parquet', '2021011814_2026011001_E.parquet']


   ✓ Loaded  2021011814_2026011001_E.parquet                     →  shape (2507555, 30)
   ✓ Loaded  2021011814_2026011001_EA.parquet                    →  shape (54844, 24)
   ✓ Loaded  2021011814_2026011001_EAL.parquet                   →  shape (201117, 29)

── Asset 1F (ccfff937-ff8c-4f84-a31c-a41f00fa0ff8) ──
   Files in events folder : ['ccfff937-ff8c-4f84-a31c-a41f00fa0ff8.parquet', '2021011814_2026011001_EAE.parquet', '2021011814_2026011001_EA.parquet', '2021011814_2026011001_EALE.parquet', '2021011814_2026011001_EAL.parquet', '2021011814_2026011001_E.parquet']
   ✓ Loaded  2021011814_2026011001_E.parquet                     →  shape (138249, 30)
   ✓ Loaded  2021011814_2026011001_EA.parquet                    →  shape (44033, 24)
   ✓ Loaded  2021011814_2026011001_EAL.parquet                   →  shape (19238, 29)

── Asset 1H (f84c869c-4c09-4485-9cdd-35a994fc2350) ──
   Files in events folder : ['2021011814_2026011001_EAE.parquet', 'f84c869c-4c09-4485-9cdd-35a994fc2350.parque

### 7a. Events Data — Preview

In [12]:
for key, df_ev in events_data.items():
    print(f"\n── {key} ──")
    print(f"   Shape   : {df_ev.shape}")
    print(f"   Columns : {df_ev.columns.tolist()}")
    display(df_ev.head(3))


── 1A_ChangeEvents ──
   Shape   : (2507555, 30)
   Columns : ['Action', 'Actor', 'AreaName', 'AlarmLimit', 'Block', 'Category', 'ConditionName', 'Description', 'EventID', 'Flags', 'LocalTime', 'LocationFullName', 'LocationTagName', 'PrevValue', 'Priority', 'ReceivedDelay', 'ServerName', 'ShelvedReason', 'Source', 'SourceParameter', 'Station', 'Time', 'TransactionID', 'Units', 'Value', 'VT_Start', 'H', 'TagID', 'AlarmStatus', 'S']


,Action,Actor,AreaName,AlarmLimit,Block,Category,ConditionName,Description,EventID,Flags,...,Station,Time,TransactionID,Units,Value,VT_Start,H,TagID,AlarmStatus,S
0,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214430,NaN,...,None,1974-03-17 20:27:01.115608,31214430,None,None,2021-10-11 10:31:51.560800,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
1,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214424,NaN,...,None,1974-03-17 20:27:01.116026,31214424,None,None,2021-10-11 10:31:51.602600,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
2,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214423,NaN,...,None,1974-03-17 20:27:01.116028,31214423,None,None,2021-10-11 10:31:51.602800,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01



── 1A_AlarmEvents ──
   Shape   : (54844, 24)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,Limit,LocationFullName,LocationTagName,AlarmStatus,VT_Start,TagID,EventTypeID,VT_End,H,S
0,S1-PROD-ESVT01,61059050,CHANGE,16,None,None,INACTIVE,INACTIVE,PTEXECST,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:48.203000,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01
1,S1-PROD-ESVT01,61059051,None,16,None,None,None,None,PTEXECST ACTIVE INACTIVE,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:48.203000,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01
2,S1-PROD-ESVT01,61059053,CHANGE,16,None,None,INACTIVE,INACTIVE,PTEXECST,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:55.256300,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01



── 1A_AlarmEventsLinked ──
   Shape   : (201117, 29)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'LastEvent', 'NextEvent', 'rowid', 'Seconds', 'PrevSeconds', 'H', 'S']


...

,Action,Actor,AreaName,AlarmLimit,Block,Category,ConditionName,Description,EventID,Flags,...,Station,Time,TransactionID,Units,Value,VT_Start,H,TagID,AlarmStatus,S
0,None,None,3I,NaN,None,14,None,PROCESS EVNT RCVRY 3I ...,31214683,NaN,...,None,1974-03-17 20:27:02.464027,31214683,None,None,2021-10-11 10:34:06.402700,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
1,None,None,3I,NaN,None,14,None,PROCESS EVNT RCVRY 3I ...,31214681,NaN,...,None,1974-03-17 20:27:02.464032,31214681,None,None,2021-10-11 10:34:06.403200,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
2,None,None,3I,NaN,None,14,None,PROCESS EVNT RCVRY 3I ...,31214682,NaN,...,None,1974-03-17 20:27:02.464033,31214682,None,None,2021-10-11 10:34:06.403300,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01



── 3I_AlarmEvents ──
   Shape   : (56023, 24)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,Limit,LocationFullName,LocationTagName,AlarmStatus,VT_Start,TagID,EventTypeID,VT_End,H,S
0,S1-PROD-ESVT01,39053520,None,16,None,None,None,None,UT ALM RECOV ...,None,...,NaN,/Assets/TRAIN_2/3I,3I,None,2022-03-24 12:35:31.853400,$UNITOPS,4,None,2022_03_24_12,S1-PROD-ESVT01
1,S1-PROD-ESVT01,39053521,None,16,None,None,None,None,UT RECOV CMP ...,None,...,NaN,/Assets/TRAIN_2/3I,3I,None,2022-03-24 12:35:31.853400,$UNITOPS,4,None,2022_03_24_12,S1-PROD-ESVT01
2,S1-PROD-ESVT01,74400276,CHANGE,16,None,None,ALL,ALL,PVSRCOPT,None,...,NaN,/Assets/TRAIN_2/3I,3I,None,2024-12-05 11:48:00.436300,01CR_3H05_DI22,4,None,2024_12_05_11,S1-PROD-ESVT01



── 3I_AlarmEventsLinked ──
   Shape   : (16142, 29)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'LastEvent', 'NextEvent', 'rowid', 'Seconds', 'PrevSeconds', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,TagID,EventTypeID,VT_End,LastEvent,NextEvent,rowid,Seconds,PrevSeconds,H,S
0,S1-PROD-ESVT01,61724101,CMDDIS,16,None,None,STOP,STOP,3G122 LUBE OIL COOLER,None,...,03EM_1003,1,None,0,2,1,12,6,2023_06_20_12,S1-PROD-ESVT01
1,S1-PROD-ESVT01,61724112,CMDDIS,16,None,None,RUN,RUN,3G122 LUBE OIL COOLER,OK,...,03EM_1003,2,None,1,1,2,3793062,12,2023_06_20_12,S1-PROD-ESVT01
2,S1-PROD-ESVT01,62980123,CMDDIS,16,None,None,STOP,STOP,3G122 LUBE OIL COOLER,None,...,03EM_1003,1,None,2,2,3,14,3793062,2023_08_03_10,S1-PROD-ESVT01


## 8. Summary

In [13]:
print(f"{'='*60}")
print(f"  SUMMARY for target tag: {TARGET_TAG}")
print(f"{'='*60}")
print(f"  Related tags in sheet         : {len(df_tags)}")
print(f"  Unique assets referenced      : {len(df_asset_map)}")
print(f"  Assets found in Historian/    : {df_availability['In Historian/'].sum()}")
print(f"  Assets found in events/       : {df_availability['In events/'].sum()}")
print(f"  Historian series loaded       : {len(historian_data)}")
print(f"  Event tables loaded           : {len(events_data)}")
print(f"{'='*60}")
print()
print("Historian series:")
for k, v in historian_data.items():
    print(f"  {k:35s}  {v.shape[0]:>8,} rows")
print()
print("Event tables:")
for k, v in events_data.items():
    print(f"  {k:45s}  {v.shape[0]:>8,} rows")

  SUMMARY for target tag: 03PIC_1104
  Related tags in sheet         : 36
  Unique assets referenced      : 8
  Assets found in Historian/    : 8
  Assets found in events/       : 8
  Historian series loaded       : 30
  Event tables loaded           : 24

Historian series:
  02FI_1000.PV                         2,130,628 rows
  03TI_1408.PV                         2,169,302 rows
  03PIC_1068.PV                        2,169,275 rows
  03PIC_1104.PV                        2,169,275 rows
  03TI_1427.PV                         2,169,291 rows
  03TI_1099.PV                         2,169,306 rows
  03TI_1100.PV                         2,169,306 rows
  03TI_1101.PV                         2,169,306 rows
  03TI_1102.PV                         2,169,305 rows
  03LIC_1071.PV                        2,168,639 rows
  03FIC_1085.PV                        2,144,381 rows
  03LIC_1094.PV                        2,169,306 rows
  03LIC_1097.PV                        2,169,291 rows
  03FI_1105.PV         

## 9. Combine All Events and Save

Concatenate all event tables (`_E`, `_EA`, `_EAL`) for all assets into a single DataFrame,
add a `source_table` column to track the origin, sort by `VT_Start`, and save to
`DATA/combined_events/<TARGET_TAG>_combined_events.parquet`.

In [14]:
import os

# ── Output directory ────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(DATA_DIR, 'combined_events')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory : {OUTPUT_DIR}")
print()

# ── Concatenate all event DataFrames ────────────────────────────────────────────
print(f"Concatenating {len(events_data)} event tables ...\n")

parts = []
for table_key, df_part in events_data.items():
    df_part = df_part.copy()
    df_part['source_table'] = table_key          # track origin: e.g. '1E_AlarmEvents'
    parts.append(df_part)
    print(f"  + {table_key:45s}  {len(df_part):>9,} rows  |  "
          f"VT_Start range: {df_part['VT_Start'].min()}  →  {df_part['VT_Start'].max()}")

df_combined = pd.concat(parts, ignore_index=True, sort=False)

print(f"\nRaw combined shape : {df_combined.shape}")
print(f"Columns            : {df_combined.columns.tolist()}")

# ── Apply timezone offset ─────────────────────────────────────────────────────────
# Historian events are stored in UTC+5:30 (IST); plant PV data is in UTC+4 (UAE local).
# Subtract 1.5 hours to align event timestamps with the PV time series.
TIME_OFFSET = pd.Timedelta(hours=1.5)
print(f"\nApplying timezone offset: VT_Start -= {TIME_OFFSET} (UTC+5:30 → UAE local UTC+4)")
print(f"  VT_Start before: {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")
df_combined['VT_Start'] = df_combined['VT_Start'] - TIME_OFFSET
print(f"  VT_Start after : {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")

# ── Sort by VT_Start ─────────────────────────────────────────────────────────────
print(f"\nSorting by VT_Start ...")
df_combined.sort_values('VT_Start', inplace=True, ignore_index=True)

print(f"Sorted combined shape : {df_combined.shape}")
print(f"VT_Start range        : {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")
print(f"Null VT_Start rows    : {df_combined['VT_Start'].isna().sum():,}")

# ── Save ─────────────────────────────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, f"{TARGET_TAG}_combined_events.parquet")
df_combined.to_parquet(output_path, index=False)
file_size_mb = os.path.getsize(output_path) / (1024 ** 2)

print(f"\n{'='*60}")
print(f"  Saved to : {output_path}")
print(f"  Rows     : {len(df_combined):,}")
print(f"  Columns  : {len(df_combined.columns)}")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"{'='*60}")
print(f"\nsource_table breakdown:")
print(df_combined['source_table'].value_counts().to_string())

Output directory : /home/h604827/ControlActions/DATA/combined_events

Concatenating 24 event tables ...

  + 1A_ChangeEvents                                2,507,555 rows  |  VT_Start range: 2021-02-01 17:19:41.804300  →  2025-06-28 00:22:59.137300
  + 1A_AlarmEvents                                    54,844 rows  |  VT_Start range: 2021-10-02 14:30:37.668800  →  2025-06-27 18:53:34.052000
  + 1A_AlarmEventsLinked                             201,117 rows  |  VT_Start range: 2021-10-09 09:58:38.952700  →  2025-06-03 10:22:26.768300
  + 1F_ChangeEvents                                  138,249 rows  |  VT_Start range: 2021-10-03 12:00:59.953200  →  2025-06-27 20:14:25.077400
  + 1F_AlarmEvents                                    44,033 rows  |  VT_Start range: 2021-10-03 12:08:23.078500  →  2025-06-27 16:57:51.410300
  + 1F_AlarmEventsLinked                              19,238 rows  |  VT_Start range: 2021-10-03 12:00:59.953200  →  2025-06-27 20:14:25.077400
  + 1H_ChangeEvents            

## 10. Batch Processing — All Tags

Processes **all alarm-tag sheets** from the Excel workbook (from the first tag sheet through to `03PI_1655_PVHI`).

For each sheet:
- The alarm type (`PVLO`, `PVHI`, `PVLO_PVHI`, `AOA`) is inferred from the sheet name.
- Sheets with no valid `ASSET ID` values are skipped automatically.
- Generic non-alarm sheets (`Sheet`, `Sheet1`, `Sheet2`) are skipped.
- Output files are named `{TAG}_{ALARM_TYPE}_combined_events.parquet` and saved to `DATA/combined_events/`.
- Files that already exist are skipped (set `SKIP_EXISTING = False` to re-process them).


In [3]:
import re

# ── Batch config ────────────────────────────────────────────────────────────────
STOP_SHEET    = '03PI_1655_PVHI'    # last sheet to process (inclusive)
SKIP_EXISTING = True                 # set False to re-process already-saved files

# Generic sheets that do not correspond to alarm tags
SKIP_SHEETS = {'Tag List', 'Sheet', 'Sheet1', 'Sheet2'}

# Asset IDs that are invalid / placeholder values
INVALID_ASSET_IDS = {'nan', 'NO', '', 'none'}


def parse_sheet_name(sheet_name):
    """
    Infer (tag, alarm_type) from a sheet name.

    Handles both space- and underscore-separated suffixes, e.g.:
      '03TIC_1023 PVLO_PVHI'  →  ('03TIC_1023', 'PVLO_PVHI')
      '03TIC_1009_PVLO'       →  ('03TIC_1009', 'PVLO')
      '03LIC1608_PVLO'        →  ('03LIC1608',  'PVLO')
      '03LIC_1094'            →  ('03LIC_1094',  None)
    """
    # Check longer alarm types first to avoid PVLO_PVHI matching as PVLO
    for alarm_type in ('PVLO_PVHI', 'PVLO', 'PVHI', 'AOA'):
        for sep in (' ', '_'):
            suffix = sep + alarm_type
            if sheet_name.endswith(suffix):
                return sheet_name[: -len(suffix)].strip(), alarm_type
    return sheet_name.strip(), None


# ── Discover sheets to process ──────────────────────────────────────────────────
_wb = openpyxl.load_workbook(EXCEL_FILE, read_only=True, data_only=True)
_all_sheets = _wb.sheetnames
_wb.close()

_stop_idx = _all_sheets.index(STOP_SHEET) if STOP_SHEET in _all_sheets else len(_all_sheets) - 1
# Skip the very first sheet ('Tag List') and stop at STOP_SHEET (inclusive)
_candidate_sheets = [s for s in _all_sheets[1:_stop_idx + 1] if s not in SKIP_SHEETS]

print(f"Candidate sheets to process : {len(_candidate_sheets)}")
for _s in _candidate_sheets:
    _t, _a = parse_sheet_name(_s)
    _alarm_suffix = f'_{_a}' if _a else ''
    _fname = f"{_t}{_alarm_suffix}_combined_events.parquet"
    _exists = os.path.exists(os.path.join(DATA_DIR, 'combined_events', _fname))
    print(f"  {'[EXISTS]' if _exists else '        ':10s}  {_s!r:38s}  →  {_fname}")


Candidate sheets to process : 19
              '03TIC_1023 PVLO_PVHI'                  →  03TIC_1023_PVLO_PVHI_combined_events.parquet
              '03TIC_1635 PVLO'                       →  03TIC_1635_PVLO_combined_events.parquet
              '03TIC_1009_PVLO'                       →  03TIC_1009_PVLO_combined_events.parquet
              '03TIC_1009_PVHI'                       →  03TIC_1009_PVHI_combined_events.parquet
              '03LIC_1619 PVLO_PVHI'                  →  03LIC_1619_PVLO_PVHI_combined_events.parquet
              '03TIC_1635_PVHI'                       →  03TIC_1635_PVHI_combined_events.parquet
              '03FIC_1668 PVHI'                       →  03FIC_1668_PVHI_combined_events.parquet
              '03PIC_1104 PVHI'                       →  03PIC_1104_PVHI_combined_events.parquet
              '03PIC1023 PVLO_PVHI'                   →  03PIC1023_PVLO_PVHI_combined_events.parquet
              '03TIC_1145 PVLO'                       →  03TIC_1145_PVLO_combine

In [ ]:
# import traceback

# # ── Load ADNOC-B.json once (reuse asset_id_to_uuid if already loaded) ───────────
# with open(ADNOC_JSON, 'r') as f:
#     _asset_nodes = json.load(f)
# _asset_id_to_uuid = {n['Name']: {'uuid': n['Id'], 'hierarchy': n['Hierarchy']} for n in _asset_nodes}

# _hist_folders   = set(os.listdir(HISTORIAN_DIR))
# _events_folders = set(os.listdir(EVENTS_DIR))

# OUTPUT_DIR   = os.path.join(DATA_DIR, 'combined_events')
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# TIME_OFFSET  = pd.Timedelta(hours=1.5)

# _KEEP_COLS = ['Potential Input parameter', 'DCS Tag Name', 'DESCRIPTION',
#               'PARAMETER', 'IS DATA AVAILABLE?', 'ASSET ID']

# batch_summary = []

# for _sheet_name in _candidate_sheets:
#     _tag, _alarm_type = parse_sheet_name(_sheet_name)
#     _alarm_suffix  = f'_{_alarm_type}' if _alarm_type else ''
#     _out_filename  = f"{_tag}{_alarm_suffix}_combined_events.parquet"
#     _out_path      = os.path.join(OUTPUT_DIR, _out_filename)

#     # ── Skip if output already exists ─────────────────────────────────────────────
#     if SKIP_EXISTING and os.path.exists(_out_path):
#         _mb = os.path.getsize(_out_path) / (1024 ** 2)
#         print(f"[SKIP  ] {_sheet_name!r:38s}  →  already exists ({_mb:.1f} MB)")
#         batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                'Status': 'SKIPPED (exists)', 'Rows': None, 'File': _out_filename})
#         continue

#     print(f"\n{'='*72}")
#     print(f"  Sheet : {_sheet_name!r}")
#     print(f"  Tag   : {_tag}   AlarmType: {_alarm_type}   Output: {_out_filename}")
#     print(f"{'='*72}")

#     try:
#         # ── Read sheet ─────────────────────────────────────────────────────────────
#         _df_sheet = pd.read_excel(EXCEL_FILE, sheet_name=_sheet_name, header=EXCEL_HEADER_ROW)

#         if 'ASSET ID' not in _df_sheet.columns or 'DCS Tag Name' not in _df_sheet.columns:
#             print(f"  [SKIP] Required columns (ASSET ID / DCS Tag Name) not found")
#             batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                    'Status': 'SKIP (missing cols)', 'Rows': None, 'File': None})
#             continue

#         _avail_cols = [c for c in _KEEP_COLS if c in _df_sheet.columns]
#         _df_tags_b = (_df_sheet[_avail_cols]
#                       .dropna(subset=['DCS Tag Name'])
#                       .copy())
#         _df_tags_b['DCS Tag Name'] = _df_tags_b['DCS Tag Name'].astype(str).str.strip()
#         _df_tags_b['ASSET ID']     = _df_tags_b['ASSET ID'].astype(str).str.strip()
#         # Filter out invalid / blank asset IDs
#         _df_tags_b = _df_tags_b[~_df_tags_b['ASSET ID'].str.lower().isin(INVALID_ASSET_IDS)]
#         _df_tags_b = _df_tags_b.reset_index(drop=True)

#         if _df_tags_b.empty:
#             print(f"  [SKIP] No valid ASSET IDs found in sheet")
#             batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                    'Status': 'SKIP (no assets)', 'Rows': None, 'File': None})
#             continue

#         _unique_assets = sorted(_df_tags_b['ASSET ID'].unique())
#         print(f"  Asset IDs : {_unique_assets}")

#         # ── Map asset names → UUIDs ────────────────────────────────────────────────
#         _asset_map_rows = []
#         for _an in _unique_assets:
#             if _an in _asset_id_to_uuid:
#                 _info = _asset_id_to_uuid[_an]
#                 _asset_map_rows.append({'Asset Name': _an, 'UUID': _info['uuid'],
#                                         'Hierarchy': _info['hierarchy']})
#             else:
#                 print(f"    [WARN] Asset '{_an}' not found in ADNOC-B.json — skipping")
#                 _asset_map_rows.append({'Asset Name': _an, 'UUID': None, 'Hierarchy': None})
#         _df_am = pd.DataFrame(_asset_map_rows)

#         # Merge UUID back to df_tags_b
#         _df_tags_b = _df_tags_b.merge(
#             _df_am.rename(columns={'Asset Name': 'ASSET ID'})[['ASSET ID', 'UUID']],
#             on='ASSET ID', how='left')

#         # ── Load events data ───────────────────────────────────────────────────────
#         _events_data_b = {}
#         for _, _arow in _df_am.iterrows():
#             _uuid  = _arow['UUID']
#             _aname = _arow['Asset Name']
#             if not _uuid or _uuid not in _events_folders:
#                 continue
#             _aev_dir = os.path.join(EVENTS_DIR, _uuid)
#             _aev_files = os.listdir(_aev_dir)
#             for _suf, _lbl in [('_E.parquet',   'ChangeEvents'),
#                                 ('_EA.parquet',  'AlarmEvents'),
#                                 ('_EAL.parquet', 'AlarmEventsLinked')]:
#                 _m = [f for f in _aev_files if f.endswith(_suf)]
#                 if _m:
#                     _df_ev = pd.read_parquet(os.path.join(_aev_dir, _m[0]))
#                     for _ts_col in ['VT_Start', 'Time', 'timestamp']:
#                         if _ts_col in _df_ev.columns:
#                             _df_ev[_ts_col] = pd.to_datetime(_df_ev[_ts_col], format='mixed')
#                     _events_data_b[f"{_aname}_{_lbl}"] = _df_ev

#         print(f"  Event tables loaded : {len(_events_data_b)}")

#         if not _events_data_b:
#             print(f"  [SKIP] No event parquet files found for any asset")
#             batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                    'Status': 'SKIP (no events)', 'Rows': None, 'File': None})
#             continue

#         # ── Concatenate ────────────────────────────────────────────────────────────
#         _parts = []
#         for _tkey, _dfp in _events_data_b.items():
#             _dfp = _dfp.copy()
#             _dfp['source_table'] = _tkey
#             _parts.append(_dfp)

#         _df_combined = pd.concat(_parts, ignore_index=True, sort=False)

#         # Apply timezone offset (UTC+5:30 → UAE local UTC+4, i.e. subtract 1.5 h)
#         _df_combined['VT_Start'] = _df_combined['VT_Start'] - TIME_OFFSET
#         _df_combined.sort_values('VT_Start', inplace=True, ignore_index=True)

#         # ── Save ───────────────────────────────────────────────────────────────────
#         _df_combined.to_parquet(_out_path, index=False)
#         _mb = os.path.getsize(_out_path) / (1024 ** 2)
#         print(f"  ✓ Saved  {_out_filename}  ({len(_df_combined):,} rows, {_mb:.1f} MB)")
#         batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                'Status': 'OK', 'Rows': len(_df_combined), 'File': _out_filename})

#     except Exception as _ex:
#         print(f"  [ERROR] {_ex}")
#         traceback.print_exc()
#         batch_summary.append({'Sheet': _sheet_name, 'Tag': _tag, 'AlarmType': _alarm_type,
#                                'Status': f'ERROR: {_ex}', 'Rows': None, 'File': None})

# # ── Final summary ──────────────────────────────────────────────────────────────
# print(f"\n{'='*72}")
# print(f"  BATCH COMPLETE  —  {len(batch_summary)} sheets processed")
# print(f"{'='*72}")
# df_batch_summary = pd.DataFrame(batch_summary)
# display(df_batch_summary)



  Sheet : '03TIC_1023 PVLO_PVHI'
  Tag   : 03TIC_1023   AlarmType: PVLO_PVHI   Output: 03TIC_1023_PVLO_PVHI_combined_events.parquet
  Asset IDs : ['1A', '1D', '1E', '1F', '1I', '1L']
  Event tables loaded : 18
  ✓ Saved  03TIC_1023_PVLO_PVHI_combined_events.parquet  (4,587,437 rows, 119.5 MB)

  Sheet : '03TIC_1635 PVLO'
  Tag   : 03TIC_1635   AlarmType: PVLO   Output: 03TIC_1635_PVLO_combined_events.parquet
  Asset IDs : ['1A', '1E', '1F', '1H', '1O']
  Event tables loaded : 15
  ✓ Saved  03TIC_1635_PVLO_combined_events.parquet  (3,866,251 rows, 100.3 MB)

  Sheet : '03TIC_1009_PVLO'
  Tag   : 03TIC_1009   AlarmType: PVLO   Output: 03TIC_1009_PVLO_combined_events.parquet
  Asset IDs : ['1A', '1D', '1E', '1F', '1K']
  Event tables loaded : 15
  ✓ Saved  03TIC_1009_PVLO_combined_events.parquet  (3,874,588 rows, 98.3 MB)

  Sheet : '03TIC_1009_PVHI'
  Tag   : 03TIC_1009   AlarmType: PVHI   Output: 03TIC_1009_PVHI_combined_events.parquet
  Asset IDs : ['1A', '1D', '1E', '1F', '1K']
  Eve

,Sheet,Tag,AlarmType,Status,Rows,File
0,03TIC_1023 PVLO_PVHI,03TIC_1023,PVLO_PVHI,OK,4587437.0,03TIC_1023_PVLO_PVHI_combined_events.parquet
1,03TIC_1635 PVLO,03TIC_1635,PVLO,OK,3866251.0,03TIC_1635_PVLO_combined_events.parquet
2,03TIC_1009_PVLO,03TIC_1009,PVLO,OK,3874588.0,03TIC_1009_PVLO_combined_events.parquet
3,03TIC_1009_PVHI,03TIC_1009,PVHI,OK,3874588.0,03TIC_1009_PVHI_combined_events.parquet
4,03LIC_1619 PVLO_PVHI,03LIC_1619,PVLO_PVHI,OK,3021947.0,03LIC_1619_PVLO_PVHI_combined_events.parquet
5,03TIC_1635_PVHI,03TIC_1635,PVHI,OK,3866251.0,03TIC_1635_PVHI_combined_events.parquet
6,03FIC_1668 PVHI,03FIC_1668,PVHI,OK,4005637.0,03FIC_1668_PVHI_combined_events.parquet
7,03PIC_1104 PVHI,03PIC_1104,PVHI,OK,5925960.0,03PIC_1104_PVHI_combined_events.parquet
8,03PIC1023 PVLO_PVHI,03PIC1023,PVLO_PVHI,OK,4587437.0,03PIC1023_PVLO_PVHI_combined_events.parquet
9,03TIC_1145 PVLO,03TIC_1145,PVLO,OK,4401453.0,03TIC_1145_PVLO_combined_events.parquet
